[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanahdt/huji_summer_class_2026/blob/main/notebooks/day1_encoding.ipynb)
# Day 1: Encoding as Interpretation
**Date:** Sunday 21 June 2026  


```{admonition} Conceptual check — before you code
:class: tip

The first quizzes for today are simply for self-assessment. It might be good to answer these first before executing any of the code below. 

Questions open in a new tab — come back here when you're done. 

**[→ Open Day 1 Quiz](../quizpages/day1_quiz.md)**
```



# Representing Musical Information

Music, as an object, is a fairly difficult thing to represent. We can represent soundwaves, pitch, loudness, timbre, *etc
.*, but those are just components of this thing that we call "music". We have to consider the sonic, the notated, and also broader aspects of structure (and logic).

In this class, we will start by looking at notated scores, which themselves are an abstraction of "the musical object." (For example, Ryan Adams playing a Taylor Swift song might look identical on the score to Taylor Swift's version, but they are markedly different.) Even if we decide to begin with musical notation, however, we face some issues.

Let's take the "Happy Birthday" for example:

![Happy Birthday](../images/happy-birthday.png)

## With DARMS (1966–1980ish)

DARMs was a project in in the 1960s and 1970s, spearheaded by Stefan Bauer-Mengelberg. It was also called the Ford-Columbia encoding language, and was intended to make it affordable and manageable for anyone to have their own scores, and to be able to print them cheaply. It was also the hope that non-musicians would be able to encode scores, which would make the process more affordable to all. 

As such, notes were just encoded by a spatial point.

![The DARMS Grid](../images/darms_grid.png){width=50%}

So to encode something like "Happy Birthday" in DARMS, we might approach it something like this:

```
!G
!M3:4
20E.U 20SU / 21QU 20QU 23QU / 22QU 20E.U 20SU / 
21QU 20QU 24QU / 23HU 20E.U 20SU /
27QD 25QD 23QU / 22LQU 21LQU 19E.U 19SU /
25QD 23QU 24QU 23H.U //

```

There are a number of different codes and tags that can be included, ranging from key signatures to instruments to articulations. Rhythms are encoded as follows:

```
W whole 
H half 
Q quarter
E eighth 
S sixteenth 
T thirty-second 
X sixty-fourth 
Y 128th
Z 256th
```

As you can imagine, this can get a little cumbersome. Here's an example by Stephen Dydo using DARMS, with the input code below. Try to parse it as best you can:

![Dydo's DARMS Example](../images/dydo_darms_example.png)


## With MUSTRAN (1967–1980ish)

Here is what "Happy Birthday" would look like as encoded with MUSTRAN:

```
GS,K*F+,3=4,8D.,16D,/,4E,4D,4G,/,2*F,8D.,16D,/,
4E,4D,4A,/,2G,,8D.,16D,/,4D+,4B,4G,/,4*F,4E,8D.,16D,/,
4B,4G,4A,/,2H.G//,END

```

MUSTRAN was created with the goal of encoding non-Western musics, and was inherently extensible. Notice here how one can encode indeterminate pitches, breath marks, and microtones.

![MUSTRAN Example](../images/wenker_p497.png)

## Humdrum (1985–present-ish)

Here is how we might encode it in the commonly used kern format, for use with the [Humdrum Toolkit:](https://www.humdrum.org/)

```
!!!OTL: Happy Birthday
**kern
*M3/4
*G:
*k[f#]
L8.d
16dJ
=
4e
4d
4g
=
2f#
L8.d
16dJ
=
4e
4d
4a
=
2g
L8.d
16dJ
=
4dd
4b
4g
=
(4f#
4e)
L8.c
16cJ
=
4b
4g
4a
=
2g.
==
*-
```

Some points worth mentioning:

- There is a distinction between metadata and musical data (the exclamation marks serve as comments of sorts).
- Notice how pitches are given specific octaves: `d` is seperate from `dd` which is separate from `D`.
- Pitches are grouped with `L` and `J` to indicate beaming, and slurs are indicated with parentheses. 
- There are score-wide elements, such as the meter (`*M3/4`), and a key signature (`*k[f#]`).
- There is also a syntax for providing musical data that might not actually exist in the score. For example, with a key signature of one sharp, it could be G major, but it could also be in E minor. So an explicit mentioning of the key is helpful.
- Perhaps more abstractly, notice that this is a two dimensional score of sorts: every moment of time is a new line, meaning one can search by line to find specific instances. This is particularly useful with _polyphonic_ music.  

You can try this out at the [Verovio Humdrum Viewer Website](https://verovio.humdrum.org/)


# Counting Pitches

For the discussion post this week, you will be asked to encode a melody and count events in that melody. We can look at it with three specific ways:

1. Humdrum, which is a series of UNIX tools that allow you to conduct analysis by piping together specific tools.
2. HumdrumR, which is a newer updating of the original Humdrum but using R. 
3. music21, which might be of interest to some.


---
## Part 1: Reading a Kern File

The encoding is the first analytical act. Read one file as plain text
before extracting anything from it.


In [ ]:
import requests, zipfile
from pathlib import Path
from collections import Counter
from itertools import islice

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from music21 import converter, note, interval

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 4)
print('Imports OK.')

In [ ]:
CORPUS_DIR = Path('beregovski_corpus')
KERN_DIR = CORPUS_DIR / 'kern'

if KERN_DIR.exists() and len(list(KERN_DIR.glob('*.krn'))) > 0:
    print(f'Corpus ready: {len(list(KERN_DIR.glob("*.krn")))} files.')
else:
    print('Downloading corpus from GitHub...')
    CORPUS_DIR.mkdir(exist_ok=True)
    r = requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp = CORPUS_DIR / 'repo.zip'
    zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z: z.extractall(CORPUS_DIR)
    src = list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists(): shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0], KERN_DIR)
        zp.unlink()
    print(f'Done. {len(list(KERN_DIR.glob("*.krn")))} kern files ready.')

In [ ]:
KERN_DIR = Path('beregovski_corpus/kern')
kern_files = sorted(KERN_DIR.glob('*.krn'))
print(f'File: {kern_files[0].name}')
print('=' * 50)
print(kern_files[0].read_text())

### Quick kern reference

| Symbol | Meaning |
|--------|---------|
| `!!!` | Global metadata record |
| `**kern` | Spine type |
| `*M6/8` | Meter |
| `*G:` | Key of G |
| `4g` | Quarter note G4 |
| `gg` | G5 (repeated letter = octave up) |
| `=1` | Barline |
| `*-` | End of spine |

```{note}
All tunes are notated in G regardless of original performance pitch.
This means pitch class data and scale-degree data are equivalent throughout the course.
```


---
## Part 2: Loading the Full Corpus

The `(df, streams)` pair is our core data structure throughout all 8 sessions.


In [ ]:
def load_corpus(kern_dir=KERN_DIR, verbose=True):
    pc2d = {7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records, sdict = {}, {}
    files = sorted(Path(kern_dir).glob('*.krn'))
    for i, f in enumerate(files):
        if verbose and i % 50 == 0: print(f'  {i+1}/{len(files)}...')
        try:
            s = converter.parse(str(f))
            ns = [n for n in s.flat.notes if isinstance(n, note.Note)]
            pcs = [n.pitch.pitchClass for n in ns]
            records[f.stem] = {
                'tune_id': f.stem, 'n_notes': len(ns),
                'pitches': [n.nameWithOctave for n in ns],
                'pitch_classes': pcs,
                'scale_degrees': [pc2d.get(p, 0) for p in pcs],
                'intervals': [interval.Interval(ns[j], ns[j+1]).semitones
                               for j in range(len(ns)-1)]
            }
            sdict[f.stem] = s
        except: pass
    if verbose: print(f'Loaded {len(records)} tunes.')
    return pd.DataFrame(records.values()), sdict

def get_ngrams(seq, n):
    return list(zip(*[islice(seq, i, None) for i in range(n)]))

print('Helper functions defined.')

In [ ]:
print('Loading corpus...')
df, streams = load_corpus()
try:
    meta = pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df = df.merge(meta, on='tune_id', how='left')
    print(f'Metadata joined. {len(df)} tunes, columns: {df.columns.tolist()}')
except Exception as e:
    print(f'Metadata not loaded ({e}). Proceeding without mode labels.')

In [ ]:
# Basic corpus statistics
print(f'Tunes: {len(df)}')
print(f'Total notes: {df["n_notes"].sum():,}')
print(f'Mean notes/tune: {df["n_notes"].mean():.1f}')
df[['tune_id','n_notes']].head(10)

In [ ]:
# Pitch class profile — full corpus
all_pcs = [pc for pcs in df['pitch_classes'] for pc in pcs]
pc_counts = Counter(all_pcs)
total = sum(pc_counts.values())
pc_profile = [pc_counts.get(i,0)/total for i in range(12)]
pc_names = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']

fig, ax = plt.subplots()
ax.bar(pc_names, pc_profile, color='steelblue', edgecolor='white')
ax.set_xlabel('Pitch class')
ax.set_ylabel('Proportion')
ax.set_title('Pitch class profile — full corpus')
plt.tight_layout(); plt.show()
print('Which pitch class is most common, and why?')

---
## Day 1 Exercise: Encoding Audit

```{admonition} Exercise
Pick one tune. Read its kern file as plain text.
Identify **three encoding decisions** the kern file makes that a performer would not need to make.
For each: (1) what musical information is *fixed*, and (2) what is *left out*?
```


In [ ]:
# Choose a tune ID from the list below
print('Available tune IDs:')
print(df['tune_id'].tolist()[:20], '...')


In [ ]:
MY_TUNE = df['tune_id'].iloc[0]  # <-- change this
kern_file = KERN_DIR / f'{MY_TUNE}.krn'
print(f'Raw kern: {MY_TUNE}')
print('=' * 50)
print(kern_file.read_text())

### Your encoding audit

**Tune chosen:** *(fill in)*

**Decision 1:** What it fixes: / What it leaves out:

**Decision 2:** What it fixes: / What it leaves out:

**Decision 3:** What it fixes: / What it leaves out:


---
## Project Log — Entry 1

> *My corpus is [X — a subset: mode / genre / region / instrument].*  
> *My research question is [Y].*  
> *I expect [Z] because [musical or theoretical reasoning].*  
> *One encoding decision I would have made differently is [X] because [reason].*

*(Write here — 100–150 words)*


---
## Preview: Day 2 (Monday 22 Jun)

**Optional preparation:** Visit https://shanahdt.github.io/mode_in_klezmer/ and listen to
one freygish tune and one minor tune. Before looking at any data, write down what you
hear as the difference between them.
